# LAB 09 — Named Entity Recognition and Information Extraction

Notebook trình bày từng mục riêng theo W9.pdf. Chạy từ thư mục `lab9`; dữ liệu trong `data/`, các bảng CSV trong `result/`.

## 1. Role — Vai trò của bài lab

Bài lab thực hành Named Entity Recognition (NER): gán nhãn loại thực thể cho từng token trong câu, rồi dùng các nhãn đó để trích xuất thông tin.

## 2. Learning Outcomes — Mục tiêu học tập

Phân biệt sequence classification với token classification; hiểu BIO và ranh giới entity; căn chỉnh nhãn word/subword; huấn luyện baseline và Transformer; đánh giá exact entity và dùng kết quả NER cho information extraction.

## 3. Dataset — CoNLL-2003

Dữ liệu tiếng Anh gồm các nhãn `O`, `PER`, `ORG`, `LOC`, `MISC` theo BIO. Các split Parquet nằm trong `data/conll2003/`.

In [1]:
from baseline_ner import load_splits, LABELS
data = load_splits()
[(split, len(rows), sum(len(r["words"]) for r in rows)) for split, rows in data.items()], LABELS

D:\NLP_exercise\lab6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


([('train', 14041, 203621),
  ('validation', 3250, 51362),
  ('test', 3453, 46435)],
 ['O',
  'B-PER',
  'I-PER',
  'B-ORG',
  'I-ORG',
  'B-LOC',
  'I-LOC',
  'B-MISC',
  'I-MISC'])

Test split chỉ dùng để báo cáo kết quả cuối; validation dùng chọn epoch tốt nhất. Dataset có 14,041 câu train, 3,250 validation và 3,453 test.

## 4. Sequence Classification vs Token Classification

Sequence classification cho một nhãn cho cả câu (ví dụ sentiment). Token classification dự đoán một nhãn ở từng vị trí token; NER thuộc loại này vì mỗi từ có thể mang nhãn BIO riêng.

## 5. BIO Tagging

`B-TYPE` đánh dấu từ đầu entity mới, `I-TYPE` đánh dấu phần tiếp theo của entity cùng loại, `O` là từ không thuộc entity. TYPE trong corpus là PER, ORG, LOC hoặc MISC.

## 6. Calculation Exercise 1 — BIO

Câu `Elon Musk founded SpaceX in California.` được gán:

`Elon/B-PER, Musk/I-PER, founded/O, SpaceX/B-ORG, in/O, California/B-LOC, ./O`.

Mỗi entity mới bắt đầu bằng B; token tiếp nối cùng entity dùng I.

## 7. Calculation Exercise 2 — Entity Boundary

Nếu toàn cụm `University of California Berkeley` là một tổ chức, nhãn là:

`University/B-ORG, of/I-ORG, California/I-ORG, Berkeley/I-ORG`.

BIO biểu diễn cả điểm bắt đầu lẫn các từ nối tiếp của một span.

## 8. Subword Tokenization

Tokenizer có thể tách một từ hiếm thành nhiều subword, trong khi nhãn CoNLL gắn với word ban đầu. Vì vậy cần giữ ánh xạ từ subword về word và không nhân nhãn của một word lên mọi vị trí subword.

## 9. Label Alignment

Dùng chiến lược first-subword: subword đầu nhận nhãn BIO của word; subword tiếp theo không tính loss. Khi quy đổi dự đoán về word level, lấy dự đoán tại subword đầu.

In [2]:
from baseline_ner import get_tokenizer, tokenize_and_align

tokenizer = get_tokenizer()
words = ["Washington", "announced", "agreement"]
labels = [5, 0, 0]  # B-LOC, O, O
enc, word_ids, aligned = tokenize_and_align(tokenizer, words, labels)
[(tokenizer.convert_ids_to_tokens(t), w, y) for t, w, y in zip(enc["input_ids"], word_ids, aligned)]

[('[CLS]', None, -100),
 ('washington', 0, 5),
 ('announced', 1, 0),
 ('agreement', 2, 0),
 ('[SEP]', None, -100)]

## 10. Ignore Index

`-100` là `IGNORE_INDEX` trong PyTorch: vị trí này bị bỏ qua khi tính cross-entropy. Special tokens, padding và subword phụ vẫn nằm trong input nhưng không tạo loss.

## 11. Calculation — Label Alignment

Nếu `Washington` bị chia thành `Wash` và `##ington`, với nhãn word là `B-LOC`, alignment là `B-LOC, -100`. Special token cũng nhận `-100`; word kế tiếp nhận nhãn riêng của nó.

## 16. Core Implementation

Pipeline: đọc Parquet → baseline Logistic Regression với word features → tokenize câu → căn nhãn BIO theo subword → fine-tune BERT-Tiny → khôi phục nhãn word-level → chuyển BIO thành span và tính exact-span metrics.

## 17. Experiment 1 — Tokenization Analysis

Thống kê trên 20 câu test; so sánh câu ngắn/dài và có/không có entity. Bảng hiển thị số từ, subword, số từ bị tách và tỷ lệ subword trên word.

In [3]:
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
analysis = pd.read_csv("result/tokenization_analysis.csv")
summary = pd.read_csv("result/tokenization_summary.csv")
display(analysis)
display(summary)
ax = summary.plot.bar(x="group", y="mean_subwords_per_word", legend=False, color="steelblue", figsize=(7,4))
ax.set_ylabel("Mean subwords per word")
ax.set_title("Experiment 1: tokenizer fragmentation by group")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

,sentence_id,group,words,tokens_ex_special,subword_splits,ratio_subwords_words,has_entity,sentence
0,1,short/entity,2,4,2,2.000000,True,Nadim Ladki
1,2,short/entity,6,12,2,2.000000,True,"AL-AIN , United Arab Emirates 1996-12-06"
2,15,short/entity,9,10,1,1.111111,True,Bitar pulled off fine saves whenever they did .
3,23,short/entity,2,6,1,3.000000,True,ROME 1996-12-06
4,24,short/entity,4,7,2,1.750000,True,Italy recalled Marcello Cuttitta
5,51,short/non-entity,7,9,1,1.285714,False,freestyle skiing moguls competition on Friday :
6,78,short/non-entity,2,3,1,1.500000,False,Scorers :
7,81,short/non-entity,4,6,1,1.500000,False,"Attendance : 10,000 ."
8,84,short/non-entity,3,6,2,2.000000,False,"Attendence : 3,000"
9,86,short/non-entity,8,8,0,1.000000,False,"for , goals against , points ) :"


,group,sentences,mean_words,mean_subword_tokens,mean_word_splits,mean_subwords_per_word
0,short,10,4.7,7.1,1.3,1.714683
1,long,10,19.5,21.5,1.0,1.071208
2,entity,10,15.0,18.5,1.7,1.549627
3,non-entity,10,9.2,10.1,0.6,1.236264


C:\Users\DELL\AppData\Local\Temp\ipykernel_20208\2147723928.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Nhận xét Experiment 1

Trong mẫu 20 câu, nhóm có entity trung bình **1.550 subword/word**, còn nhóm không entity **1.236**. Mẫu câu ngắn có tỷ lệ **1.715**, dài **1.071**. Đây là kết quả của mẫu đã chọn, không khẳng định xu hướng cho toàn corpus; alignment giúp giữ nhãn word-level sau khi chia subword.